In [ ]:
!pip install "swig"
!pip install "stable-baselines3==2.0.0a5"
!pip install "gymnasium[box2d]"
!pip install 'shimmy>=0.2.1'

In [ ]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import DummyVecEnv, make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

In [ ]:
# Create the environment
np.random.seed(0)
env = gym.make("LunarLander-v2")

In [ ]:
env = make_vec_env('LunarLander-v2', n_envs=16)

In [ ]:
# We use MultiLayerPerceptron (MLPPolicy) because the input is a vector,
# We added some parameters to accelerate the training
model = PPO(
    policy = 'MlpPolicy',
    env = env,
    n_steps = 1024,
    batch_size = 64,
    n_epochs = 4,
    gamma = 0.999,
    gae_lambda = 0.98,
    ent_coef = 0.01,
    verbose=1)


In [ ]:
# Train it for 1,000,000 timesteps
model.learn(total_timesteps=1000000)
# Save the model
model_name = "ppo-LunarLander-v2"
model.save(model_name)

In [ ]:
eval_env = gym.make("LunarLander-v2")
# Evaluate the model with 10 evaluation episodes and deterministic=True
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"mean_reward={mean_reward:.2f} +/- {std_reward}")

Trained Agent

In [ ]:
# The trained agent
trained_env = gym.make("LunarLander-v2", render_mode="rgb_array")
obs = trained_env.reset()
states8, other = obs
trained_obs = []
for i in range(1000):
    action, _states = model.predict(states8)
    obs, rewards, terminated, truncated, info = env.step(action)
    trained_obs.append(obs)
    env.render()
print('Trained observations: ',trained_obs)



In [ ]:
import pandas as pd
import numpy as np

observation_df = pd.read_csv('trained_observations_all.csv')
for index, obs in observation_df.iterrows():
    obs_array = obs.to_numpy()
    print(obs_array)

In [ ]:
import os
import gym
import matplotlib.pyplot as plt
os.environ["IMAGEIO_FFMPEG_EXE"] = r"MyFFMPEG_PATH"
from IPython.display import clear_output

# env = gym.make("LunarLander-v2", render_mode="rgb_array")
# env.action_space.seed(42)
# observation = env.reset()
# print(observation)
# obs, other = observation
# print(obs)

for index, obs in observation_df.iloc[:100, :].iterrows():
    obs_array = obs.to_numpy()
    action, _states = model.predict(obs_array)
    print('action', action)
    observations, reward, terminated, truncated, info = env.step(action)

    if terminated or truncated:
        # Reset the environment
        print("Environment is reset")
        observation, info = env.reset()
        
#     clear_output(wait=True)
#     plt.imshow( env.render() )
#     plt.show()

env.close()



In [ ]:
import os
import gym
import matplotlib.pyplot as plt
os.environ["IMAGEIO_FFMPEG_EXE"] = r"MyFFMPEG_PATH"
from IPython.display import clear_output

# env = gym.make("LunarLander-v2", render_mode="rgb_array")
# env.action_space.seed(42)
observation = env.reset()
print(observation)
obs, other = observation
print(obs)

for _ in range(1000):
    action, _states = model.predict(obs)
    print('action', action)
    observations, reward, terminated, truncated, info = env.step(action)

    if terminated or truncated:
        # Reset the environment
        print("Environment is reset")
        observation, info = env.reset()
        
    clear_output(wait=True)
    plt.imshow( env.render() )
    plt.show()

env.close()

GIF

In [ ]:
import imageio
import numpy as np
images = []
# obs = model.env.reset()
img = model.env.render(mode='rgb_array')
for i in range(350):
    images.append(img)
    action, _ = model.predict(obs)
#     obs, _, _ , _ = model.env.step(action)
    observations, reward, terminated, truncated, info = env.step(action)
    img = model.env.render(mode='rgb_array')

imageio.mimsave('lander_ppo.gif', [np.array(img) for i, img in enumerate(images) if i%2 == 0], duration=29)

Video


In [ ]:
import os
import gym
from gym.wrappers.record_video import RecordVideo

os.environ["IMAGEIO_FFMPEG_EXE"] = r"MyFFMPEG_PATH"
# Create the Lunar Lander environment


# Specify the directory where the videos will be saved

# Wrap the environment with the Monitor to record video
env = gym.make('LunarLander-v2', render_mode="rgb_array")
env = RecordVideo(env, './video',  episode_trigger = lambda episode_number: True)
observation = env.reset()
obs, other = observation

# Your agent training and control logic here
for _ in range(1000):
    action, _states = model.predict(obs)
#     print('action', action)
    observations, reward, terminated, truncated, info = env.step(action)

    if terminated or truncated:
        # Reset the environment
        print("Environment is reset")
        observation = env.reset()
        
# Train your agent, take actions, and render the environment to record video
# Close the environment when done
env.close()


In [ ]:
import os
os.environ["IMAGEIO_FFMPEG_EXE"] = r"MyFFMPEG_PATH"

In [ ]:
conda install -c conda-forge ffmpeg

In [ ]:
!pip install moviepy

In [ ]:
import gym
from stable_baselines3.common.vec_env import VecVideoRecorder, DummyVecEnv
import moviepy

env_id = 'LunarLander-v2'
video_folder = 'logs/videos/'
video_length = 100

env = DummyVecEnv([lambda: gym.make(env_id, render_mode = 'rgb_array')])

obs = env.reset()

# Record the video starting at the first step
env = VecVideoRecorder(env, video_folder,
                       record_video_trigger=lambda x: x == 0, video_length=video_length,
                       name_prefix="random-agent-{}".format(env_id))

env.reset()
for _ in range(video_length + 1):
  action = [env.action_space.sample()]
  obs, _, _, _ = env.step(action)
# Save the video
env.close()


In [ ]:
import PIL.Image
from pyvirtualdisplay import Display

In [ ]:
# Set up a virtual display to render the Lunar Lander environment.
Display(visible=0, size=(840, 480)).start();

In [ ]:
env.reset()
PIL.Image.fromarray(env.render(mode='rgb_array'))

In [ ]:
import os
import gym
import matplotlib.pyplot as plt
os.environ["IMAGEIO_FFMPEG_EXE"] = r"MyFFMPEG_PATH"
from IPython.display import clear_output

# env = gym.make("LunarLander-v2", render_mode="rgb_array")
# env.action_space.seed(42)

observation = env.reset()

for _ in range(1000):
    
    observation,reward, terminated, truncated, info = env.step(observation)

    if terminated or truncated:
        # Reset the environment
        print("Environment is reset")
        observation, info = env.reset()
        
    clear_output(wait=True)
    plt.imshow( env.render() )
    plt.show()

env.close()